# Tarea 2. Detección de grietas en pavimento
**Procesamiento de Imágenes, Universidad Diego Portales**

Integrantes: _______________________ · _______________________

Sigan los pasos del enunciado en orden. Completen las funciones marcadas con
`NotImplementedError` sin cambiar sus nombres ni sus firmas, y agreguen las celdas de código que
pida cada paso. Este cuaderno contiene solo el código y sus salidas: los resultados y su análisis
van en el informe en PDF.

## 0. Preparación

In [1]:
import os, subprocess

RUTA = "CrackForest-dataset"
if not os.path.isdir(RUTA):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/cuilimeng/CrackForest-dataset.git", RUTA],
                   check=True)
print("imágenes:", len([f for f in os.listdir(f"{RUTA}/image") if f.endswith(".jpg")]))
print("máscaras:", len([f for f in os.listdir(f"{RUTA}/groundTruth") if f.endswith(".mat")]))

imágenes: 155
máscaras: 118


In [2]:
import time
import cv2
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

OpenCV 5.0.0 | NumPy 2.5.2


## 1. Utilidades entregadas

**No modifiquen estas celdas.** Toda función de segmentación recibe una imagen en grises `uint8`
y devuelve una máscara `uint8` con valores en `{0, 1}` (no 0-255) y el mismo tamaño.
`evaluar_por_imagen` verifica este contrato y se detiene con un error si una máscara no lo cumple.
Varias funciones de OpenCV (`Canny`, `floodFill`, `threshold` con máximo 255) producen 0/255:
conviertan antes de devolver.

In [3]:
def cargar_par(nombre, ruta=RUTA):
    """Devuelve (imagen en escala de grises, máscara binaria) para un nombre como '001'."""
    img = cv2.imread(f"{ruta}/image/{nombre}.jpg", cv2.IMREAD_GRAYSCALE)
    mat = sio.loadmat(f"{ruta}/groundTruth/{nombre}.mat")
    seg = mat["groundTruth"][0, 0]["Segmentation"]   # 1 = fondo, 2 = grieta
    return img, (seg == 2).astype(np.uint8)


# La carpeta image/ tiene 155 archivos, pero solo 118 tienen máscara:
# los identificadores se construyen desde groundTruth/.
IDS = sorted(f[:-4] for f in os.listdir(f"{RUTA}/groundTruth") if f.endswith(".mat"))
print(f"{len(IDS)} pares imagen/máscara:", IDS[:5], "...", IDS[-3:])

118 pares imagen/máscara: ['001', '002', '003', '004', '005'] ... ['116', '117', '118']


In [4]:
def mostrar(imagenes, titulos=None, cols=4, escala=3.2, cmap="gray"):
    """Grilla de imágenes en escala de grises, binarias o RGB."""
    n = len(imagenes)
    filas = int(np.ceil(n / cols))
    fig, ax = plt.subplots(filas, cols, figsize=(escala * cols, escala * filas * 0.72))
    ax = np.atleast_1d(ax).ravel()
    for i, a in enumerate(ax):
        if i < n:
            im = imagenes[i]
            if im.ndim == 3:
                a.imshow(im)
            else:
                a.imshow(im, cmap=cmap, vmin=0, vmax=1 if im.max() <= 1 else 255)
            if titulos is not None:
                a.set_title(titulos[i], fontsize=9)
        a.axis("off")
    plt.tight_layout()
    plt.show()


def superponer(img, pred, gt):
    """Imagen RGB con la comparación: verde = acierto, rojo = falso positivo, azul = falso negativo."""
    p, g = pred.astype(bool), gt.astype(bool)
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    out[p & g] = (0, 220, 0)
    out[p & ~g] = (230, 0, 0)
    out[~p & g] = (0, 90, 255)
    return out

In [5]:
def metricas(pred, gt):
    """Métricas a nivel de píxel entre una máscara predicha y la de referencia."""
    p, g = pred.astype(bool), gt.astype(bool)
    tp = int((p & g).sum())
    fp = int((p & ~g).sum())
    fn = int((~p & g).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    return {"IoU": iou, "P": precision, "R": recall, "F1": f1}


def metricas_con_tolerancia(pred, gt, tol=2):
    """Igual que metricas(), pero admite un desajuste de hasta `tol` píxeles.

    Se cuenta como acierto todo píxel predicho que caiga dentro de la máscara de
    referencia dilatada, y como detectado todo píxel de referencia que caiga dentro
    de la predicción dilatada.
    """
    # se normaliza a {0, 1}: con una máscara 0/255 el denominador p.sum() quedaría inflado
    p, g = pred.astype(bool).astype(np.uint8), gt.astype(bool).astype(np.uint8)
    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol + 1, 2 * tol + 1))
    precision = (p & cv2.dilate(g, ee)).sum() / p.sum() if p.sum() else 0.0
    recall = (g & cv2.dilate(p, ee)).sum() / g.sum() if g.sum() else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"P_tol": precision, "R_tol": recall, "F1_tol": f1}

In [6]:
def evaluar_por_imagen(segmentador, ids=None, tol=2):
    """Aplica `segmentador(img) -> máscara binaria` a cada imagen y devuelve un
    DataFrame con una fila por imagen (índice = identificador)."""
    ids = IDS if ids is None else ids
    filas = {}
    for n in ids:
        img, gt = cargar_par(n)
        pred = segmentador(img)
        assert pred.dtype == np.uint8 and pred.shape == gt.shape and pred.max() <= 1,             f"imagen {n}: la máscara debe ser uint8, del tamaño de la imagen y con valores {{0, 1}}"
        filas[n] = {**metricas(pred, gt), **metricas_con_tolerancia(pred, gt, tol)}
    return pd.DataFrame(filas).T


def evaluar_conjunto(segmentador, ids=None, tol=2):
    """Promedio sobre el conjunto de todas las métricas de evaluar_por_imagen()."""
    return evaluar_por_imagen(segmentador, ids, tol).mean().to_dict()


def tabla(resultados):
    """Convierte un dict {nombre: métricas} en un DataFrame con las columnas del informe."""
    df = pd.DataFrame(resultados).T
    return df[["IoU", "P", "R", "F1", "F1_tol"]]

In [7]:
FILTROS = {
    "sin filtro":          lambda x: x,
    "promediador 5×5":     lambda x: cv2.blur(x, (5, 5)),
    "gaussiano 5×5":       lambda x: cv2.GaussianBlur(x, (5, 5), 0),
    "mediana 5×5":         lambda x: cv2.medianBlur(x, 5),
    "mediana 9×9":         lambda x: cv2.medianBlur(x, 9),
    "bilateral (9,50,50)": lambda x: cv2.bilateralFilter(x, 9, 50, 50),
}

## Parte 1. Filtrado
### Paso 1. Relación contraste-ruido

In [ ]:
def contraste_grieta_fondo(img, mascara):
    """CNR = |media(grieta) - media(fondo)| / desviacion(fondo), en punto flotante."""
    raise NotImplementedError

### Paso 2. Ruta C con filtro intercambiable

**Idea.** La grieta es un *valle* oscuro y angosto (3 a 5 px). El cierre con un elemento de
`k_bh` × `k_bh` rellena todo valle más angosto que el elemento con el nivel del asfalto que lo
rodea, así que `cierre(f) − f` (black-hat) vale casi cero en el asfalto y es alto solo sobre la
grieta. Como esa respuesta es casi toda cercana a cero, el umbral `media + k_sigma · desviación`
se queda con los valores atípicos de cada imagen. El filtro se recibe como argumento para poder
cambiar **solo** el suavizado y medir su efecto sobre la segmentación (Tabla 1); por la misma
razón no lleva post-proceso.

In [ ]:
def segmentar_blackhat_filtro(img, filtro, k_bh=21, k_sigma=3.0):
    """filtro(img) -> black-hat elíptico k_bh x k_bh -> umbral media + k_sigma * desviación. Sin post-proceso."""
    raise NotImplementedError

### Paso 3. Tabla 1 y recorte ampliado

In [ ]:
# Tabla 1


In [ ]:
# recorte ampliado bajo los seis filtros

## Parte 2. Tres rutas de segmentación
### Paso 4. Ruta A, Otsu

In [ ]:
def segmentar_otsu(img, k_mediana=5):
    """Mediana k_mediana x k_mediana y umbral de Otsu inverso."""
    raise NotImplementedError

### Paso 5. Ruta B, Canny, cierre y relleno

In [ ]:
def segmentar_canny(img, k_blur=5, t1=50, t2=150, k_cierre=5):
    """Ruta B: suavizado gaussiano, Canny, cierre elíptico y relleno del interior.

    Canny marca los dos bordes de la grieta, no la grieta. El cierre sella el
    contorno y el relleno por inundación desde una esquina, negado, recupera las
    regiones que quedaron encerradas.
    """
    # --- Completar: gaussiano k_blur x k_blur, Canny(t1, t2) y cierre elíptico k_cierre x k_cierre.
    # `contorno` debe quedar como imagen 0/255 con los bordes cerrados. Borren el raise al terminar.
    contorno = None
    raise NotImplementedError

    # --- Entregado: relleno del interior (no modificar) ---
    inundado = contorno.copy()
    h, w = contorno.shape
    cv2.floodFill(inundado, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 255)
    interior = cv2.bitwise_not(inundado)
    return ((contorno > 0) | (interior > 0)).astype(np.uint8)

### Paso 6. Ruta C, black-hat

In [ ]:
def segmentar_blackhat(img, k_mediana=5, k_bh=21, k_sigma=3.0):
    """segmentar_blackhat_filtro con mediana k_mediana x k_mediana."""
    raise NotImplementedError

### Paso 7. Tabla 2, fracción marcada y figura

In [ ]:
# Tabla 2

In [ ]:
# fracción de la imagen marcada por cada ruta

In [ ]:
# figura con superponer()

## Parte 3. Post-proceso morfológico
### Paso 8. `postprocesar`

In [ ]:
def postprocesar(mascara, k_cierre=0, area_min=0, orden="cierre-area"):
    """Cierre morfológico y eliminación de componentes pequeñas.

    - Cierre con elemento elíptico de tamaño k_cierre.
    - Eliminación de componentes de área menor a area_min, usando
      cv2.connectedComponentsWithStats con conectividad 8.
    - orden = "cierre-area" o "area-cierre" fija qué etapa se aplica primero.

    k_cierre <= 1 o area_min <= 0 desactivan la etapa correspondiente.
    """
    def cierre(b):
        raise NotImplementedError

    # --- Entregado: filtro de área (no modificar) ---
    def area(b):
        if area_min <= 0:
            return b
        _, etiquetas, stats, _ = cv2.connectedComponentsWithStats(b, connectivity=8)
        conservar = np.where(stats[1:, cv2.CC_STAT_AREA] >= area_min)[0] + 1
        return np.isin(etiquetas, conservar).astype(np.uint8)

    # aplicar cierre() y area() en el orden indicado por `orden`;
    # cualquier otro valor de `orden` debe lanzar ValueError
    raise NotImplementedError

### Paso 9. Tabla 3 y figura

In [ ]:
# Tabla 3

In [ ]:
# figura con las cinco configuraciones de la ruta C

## Parte 4. Pipeline final
### Paso 10. `pipeline_final`, Tabla 4 y gráfico

In [ ]:
def pipeline_final(img):
    """Mejor filtro (Tabla 1) -> black-hat -> postprocesar con cierre(9) y área(50), en ese orden."""
    raise NotImplementedError

In [ ]:
# Tabla 4

In [ ]:
# gráfico de barras

### Paso 11. Peores y mejores casos

In [ ]:
# tres peores y tres mejores casos según F1